# Polar PPG hypnodensity and Captum interpretation

This notebook loads Polar Verity Sense **PPG_0**, defaults to the existing **0.5-8 Hz
Butterworth filter with sign reversal**, and runs the same pinned PPG-only wav2sleep
checkpoint as `polar.ipynb`. Raw ACC, filtered PPG, and hypnodensity share timestamps.

For selected 30-second output epochs, Captum computes **Integrated Gradients (IG)**
and **Occlusion**. These explain model score dependence, not physiological causation
or the accuracy of sleep staging. There are no reference sleep labels here.

Use the **Python (infraslow)** kernel. Install optional dependencies in that environment
from the repository root with `python -m pip install -r offline_processing/requirements-captum.txt`.
Input paths come from the existing Git-ignored `paths.local.json` or
`OVERNIGHT_VERITY_SENSE_PATH`. Recordings and generated outputs remain local.


In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import sys

import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd
import torch
from IPython import get_ipython
from IPython.display import display
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
from wav2sleep import load_model

PROCESSING = Path.cwd() if (Path.cwd() / "notebook_paths.py").exists() else Path.cwd() / "offline_processing"
PROCESSING = PROCESSING.resolve()
if str(PROCESSING) not in sys.path:
    sys.path.insert(0, str(PROCESSING))
from notebook_paths import input_path
from polar_interpretability import (
    MODEL_URI, MODEL_REVISION, MODEL_FS, STAGES, load_polar_recording,
    prepare_input, epoch_table, candidate_epochs, explain_epoch,
    aggregate_attributions, plot_overview, plot_explanations, plot_pulse_detail,
)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)


## Recording and model settings

`PPG_INPUT = "filtered"` is the default. `"raw"` reproduces the previous raw-input
experiment. Sampling rate 55 Hz is used **only for the existing filter**; resampling
uses the recorded sensor timestamps. Timestamp gaps over one second and invalid or
duplicate PPG timestamps are rejected before filtering to avoid bridging dropouts.

Upstream preprocessing interpolates to **1,024 samples per 30 s (34.1333 Hz)**,
zero-pads to ten hours, and globally z-scores **including padding**. Outputs after
the recording are removed; the final partial epoch is flagged and excluded from
attribution candidates. No beats or HRV measures are supplied to the model.


In [ ]:
PPG_INPUT = "filtered"  # "filtered" or "raw"
FS_PPG = 55
MAX_HOURS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_THREADS = 4
MODEL_CACHE = os.environ.get("WAV2SLEEP_CACHE_DIR")  # Optional existing Hugging Face cache.

torch.set_num_threads(TORCH_THREADS)
torch.manual_seed(0)
raw_ppg, acc = load_polar_recording(input_path("verity_sense_path"))
filtered_ppg = pd.Series(
    -nk.signal_filter(raw_ppg.to_numpy(), sampling_rate=FS_PPG, lowcut=0.5, highcut=8,
                      order=3, method="butterworth"),
    index=raw_ppg.index, name="PPG_0_filtered",
)
if PPG_INPUT not in {"raw", "filtered"}:
    raise ValueError('PPG_INPUT must be "raw" or "filtered".')
selected_ppg = filtered_ppg if PPG_INPUT == "filtered" else raw_ppg
start = raw_ppg.index[0]
output_dir = PROCESSING / "outputs" / "polar_hypnodensity" / f"{start:%Y%m%dT%H%M%S}_{PPG_INPUT}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"{len(raw_ppg):,} PPG samples; {start} to {raw_ppg.index[-1]}; model input={PPG_INPUT}")


In [ ]:
full_input, duration_s, observed_samples = prepare_input(selected_ppg, max_hours=MAX_HOURS)
full_input = full_input.to(DEVICE)
model = load_model(MODEL_URI, revision=MODEL_REVISION, device=DEVICE, cache_dir=MODEL_CACHE)
model.eval()
if model.num_classes != 4 or "PPG" not in model.valid_signals or model.signal_encoders.causal:
    raise ValueError("Expected the non-causal, four-class cardio-respiratory checkpoint.")
# Keep tensors usable for later gradients; inference_mode is deliberately not used.
with torch.no_grad():
    logits = model({"PPG": full_input})
    probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()
hypnodensity = epoch_table(probabilities, start, duration_s)
hypnodensity.to_csv(output_dir / "hypnodensity.csv")
print(f"{len(hypnodensity)} epochs; {int((~hypnodensity['is_complete_epoch']).sum())} partial; device={DEVICE}")
display(hypnodensity.head())


In [ ]:
overview = plot_overview(raw_ppg, filtered_ppg, acc, hypnodensity, input_name=PPG_INPUT)
overview.savefig(output_dir / "overview.png", dpi=140)
display(overview)
plt.close(overview)
del overview  # Release long-night plotting arrays before gradient calculations.


## Choose output epochs to explain

The table ranks **complete epochs** by raw/filtered disagreement (when a matching
raw CSV already exists), then by prediction entropy. Entropy measures uncertainty;
it is not an explanation-quality or physiological-quality score. The default runs
one example to keep full-night backward passes manageable. Set explicit zero-based
epoch numbers below to inspect confident predictions, transitions, or artifacts.

A missing raw comparison does not prevent filtered-input attribution. The comparison
is used only for selection; it never enters the model. A mismatched clock index is
rejected rather than compared row by row.


In [ ]:
RAW_COMPARISON_CSV = None  # Optional explicit Path; otherwise look for the prior raw run.
raw_csv = (Path(RAW_COMPARISON_CSV) if RAW_COMPARISON_CSV is not None else
           PROCESSING / "outputs" / "polar_wav2sleep" / f"{start:%Y%m%dT%H%M%S}_ppg0_hypnodensity.csv")
raw_comparison = pd.read_csv(raw_csv, index_col=0, parse_dates=[0]) if raw_csv.exists() else None
if raw_comparison is None:
    print("No matching raw CSV: candidates are ranked by uncertainty.")
candidates = candidate_epochs(hypnodensity, raw_comparison)
candidates.to_csv(output_dir / "candidate_epochs.csv", index=False)
display(candidates.head(12))


## Attribution settings and meaning

The wrapper returns the selected epoch's **stage logit** (or a target-minus-contrast
logit). Probabilities are shown in the hypnodensity; attribution units are logits.
Positive IG supports the selected score **relative to the chosen reference**;
negative IG opposes it. Occlusion is original score minus score after replacing a
block. Occlusion values are not additive contributions and must not be summed.

Only the observed window around the target epoch is changed. **The entire original
night, including all context outside that window and its exact normalized padding,
is passed to the model on every evaluation.** Dataset normalization stays fixed.
The model's internal normalization layers still behave normally. This is a local
counterfactual explanation conditional on the unmodified rest of the night; it does
not claim that all influential information is inside this window. Increase
`CONTEXT_SECONDS` to explore dependence on wider context.

Two local references are compared: `mean` flattens the window to its own mean;
`reversed` reverses the same samples, preserving their amplitude distribution.
Neither reference represents a known sleep state. Both can create unusual waveform
boundaries. Inspect reference sensitivity alongside the original waveform and ACC.

IG uses one integration sample per batch. The CNN is checkpointed during backward
to reduce memory without changing forward predictions. Integration steps increase
until **abs(delta) <= IG_ATOL + IG_RTOL * abs(original score - reference score)**
or `IG_MAX_STEPS` is reached. Failed convergence is saved and displayed; such an IG
map should not be treated as a reliable decomposition. A small completeness error
alone does not establish stability or physiological validity.


In [ ]:
EPOCHS_TO_EXPLAIN = None  # None selects AUTO_EXAMPLES candidates; or e.g. [100, 200].
AUTO_EXAMPLES = 1
TARGET_STAGE = None  # None = predicted class for each epoch; or "Wake", "Light", "Deep", "REM".
CONTRAST_STAGE = None  # Optional alternative class for a logit difference.
CONTEXT_SECONDS = 30  # On each side of the 30-second output epoch.
BASELINES = ["mean", "reversed"]
OCCLUSION_SECONDS = 10
IG_STEPS = 16
IG_MAX_STEPS = 128
IG_ATOL = 0.02
IG_RTOL = 0.05

epochs = (candidates["epoch"].iloc[:AUTO_EXAMPLES].astype(int).tolist()
          if EPOCHS_TO_EXPLAIN is None else list(EPOCHS_TO_EXPLAIN))
if not epochs:
    raise ValueError("Select at least one complete epoch.")
print("Explaining epochs:", epochs)
display(hypnodensity.iloc[epochs][STAGES + ["is_complete_epoch"]])


In [ ]:
explanations = {}
diagnostics_rows = []
for epoch in epochs:
    explanations[epoch] = {}
    for baseline in BASELINES:
        result = explain_epoch(
            model, full_input, epoch, observed_samples, stage=TARGET_STAGE, contrast=CONTRAST_STAGE,
            baseline=baseline, context_seconds=CONTEXT_SECONDS, occlusion_seconds=OCCLUSION_SECONDS,
            n_steps=IG_STEPS, max_steps=IG_MAX_STEPS, atol=IG_ATOL, rtol=IG_RTOL,
            checkpoint=True, duration_s=duration_s,
        )
        explanations[epoch][baseline] = result
        diagnostics_rows.append(result.diagnostics)
        saved_samples = result.samples.copy()
        saved_samples.insert(0, "time", start + pd.to_timedelta(saved_samples["seconds"], unit="s"))
        saved_samples.to_csv(output_dir / f"epoch_{epoch}_{baseline}_samples.csv", index=False)
        aggregate_attributions(result.samples).to_csv(output_dir / f"epoch_{epoch}_{baseline}_one_second.csv", index=False)
        print(f"Completed {baseline}: delta={result.diagnostics['convergence_delta']:.4g}; "
              f"converged={result.diagnostics['converged']}; {result.diagnostics['elapsed_s']:.1f} s")

diagnostics = pd.DataFrame(diagnostics_rows)
diagnostics.to_csv(output_dir / "attribution_diagnostics.csv", index=False)
display(diagnostics)
if not diagnostics["converged"].all():
    print("Some IG results did not meet the completeness tolerance. Inspect diagnostics and raise IG_MAX_STEPS before interpretation.")


In [ ]:
for epoch, results in explanations.items():
    context_figure = plot_explanations(results, start, acc, hypnodensity)
    context_figure.savefig(output_dir / f"epoch_{epoch}_context.png", dpi=140)
    display(context_figure)
    plt.close(context_figure)
    detail_figure = plot_pulse_detail(results, start)
    detail_figure.savefig(output_dir / f"epoch_{epoch}_pulse_detail.png", dpi=140)
    display(detail_figure)
    plt.close(detail_figure)


## Reading and checking the result

- The shaded interval is the **output epoch**, while the attribution window includes
  configurable surrounding observations. The full night remains in the model.
- The pulse detail is the actual resampled model input colored by per-sample IG.
  Color scales are separate for the two references; compare signs and timing, then
  inspect numerical magnitudes in the CSVs.
- Signed one-second IG values are **sums**. Absolute IG sums describe magnitude but
  discard direction. Occlusion values are score differences and are aggregated by
  **mean**, never by sum.
- Inspect both baseline choice and integration convergence. Repeated runs with more
  steps or wider windows can change the interpretation. Flat replacements can be
  out of distribution, and a time-reversed waveform is not a physiological control.
- Attribution to motion-corrupted PPG may expose artifact sensitivity. ACC provides
  context here but is never input to the PPG-only checkpoint.
- The filtered/raw experiments also differ in polarity. Separate sign reversal,
  baseline drift, and band-pass changes before attributing their prediction differences
  to a single physiological mechanism.

Methods: [Captum Integrated Gradients](https://captum.ai/api/integrated_gradients.html),
[Captum Occlusion](https://captum.ai/api/occlusion.html),
[memory guidance](https://captum.ai/docs/faq),
[pinned wav2sleep source](https://github.com/joncarter1/wav2sleep/tree/278e30463c8149c4e6899b8784da492fec695bd9).


In [ ]:
provenance = {
    "model": MODEL_URI, "model_revision": MODEL_REVISION, "input": PPG_INPUT,
    "filter": {"nominal_fs": FS_PPG, "band_hz": [0.5, 8], "order": 3, "sign_reversed": True},
    "recording_start": str(start), "duration_s": duration_s,
    "observed_model_samples": observed_samples, "padded_model_samples": int(full_input.shape[-1]),
    "preprocessing": "upstream interpolation + 10h zero-padding + global z-score including padding",
    "normalization_fixed_during_attribution": True, "context_seconds": CONTEXT_SECONDS,
    "baseline_names": BASELINES, "epochs": epochs, "stage": TARGET_STAGE, "contrast": CONTRAST_STAGE,
    "occlusion_seconds": OCCLUSION_SECONDS, "ig_initial_steps": IG_STEPS, "ig_max_steps": IG_MAX_STEPS,
    "ig_atol": IG_ATOL, "ig_rtol": IG_RTOL,
    "all_ig_converged": bool(diagnostics["converged"].all()),
    "versions": {name: importlib.metadata.version(name) for name in
                 ["captum", "torch", "wav2sleep", "numpy", "pandas", "neurokit2"]},
}
(output_dir / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print("Saved local results to", output_dir)
